<a href="https://colab.research.google.com/github/sandyreda898-oss/Smart-Agricultural-Water-Management/blob/main/weather_soil_water_modeling/notebooks/04_et0_calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

repo_path = "/content/Smart-Agricultural-Water-Management"

if not os.path.exists(repo_path):
    !git clone https://github.com/sandyreda898-oss/Smart-Agricultural-Water-Management.git
else:
    print("Repository already exists. Using the current copy.")

Cloning into 'Smart-Agricultural-Water-Management'...
remote: Enumerating objects: 698, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 698 (delta 43), reused 0 (delta 0), pack-reused 562 (from 1)
Receiving objects: 100% (698/698), 134.59 MiB | 26.83 MiB/s, done.
Resolving deltas: 100% (282/282), done.
Updating files: 100% (447/447), done.


In [2]:
import pandas as pd
import numpy as np

weather_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/weather_daily_clean.csv.gz"
)

weather = pd.read_csv(
    weather_path,
    parse_dates=["obs_date"]
)

print("Weather shape:", weather.shape)
weather.head()

Weather shape: (136002, 17)


,point_id,obs_date,ALLSKY_SFC_SW_DWN,PRECTOTCORR,PS,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M,latitude,longitude,district_id,spatial_support,et0_inputs_complete,rainfall_available,rainfall_review_flag
0,grid_0001,2017-01-01,11.32,0.04,102.26,64.90,9.79,16.73,5.38,2.26,31.012854,30.818041,9,gridded estimate,True,True,False
1,grid_0001,2017-01-02,10.49,0.22,102.33,73.94,10.76,16.53,5.64,2.28,31.012854,30.818041,9,gridded estimate,True,True,False
2,grid_0001,2017-01-03,11.33,0.15,102.42,74.23,11.64,17.38,6.94,1.60,31.012854,30.818041,9,gridded estimate,True,True,False
3,grid_0001,2017-01-04,10.52,0.39,102.29,80.24,12.06,17.60,7.99,2.21,31.012854,30.818041,9,gridded estimate,True,True,False
4,grid_0001,2017-01-05,10.12,0.00,102.14,63.24,11.77,17.80,5.71,1.33,31.012854,30.818041,9,gridded estimate,True,True,False


In [3]:
required_columns = [
    "point_id",
    "obs_date",
    "latitude",
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "WS2M",
    "PS",
    "ALLSKY_SFC_SW_DWN"
]

missing_columns = [
    col for col in required_columns
    if col not in weather.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [4]:
eto = weather.copy()

eto["day_of_year"] = eto["obs_date"].dt.dayofyear
eto["latitude_rad"] = np.radians(eto["latitude"])

In [5]:
def saturation_vapor_pressure(temp_c):
    return 0.6108 * np.exp(
        (17.27 * temp_c) /
        (temp_c + 237.3)
    )

eto["es_tmax"] = saturation_vapor_pressure(
    eto["T2M_MAX"]
)

eto["es_tmin"] = saturation_vapor_pressure(
    eto["T2M_MIN"]
)

eto["es"] = (
    eto["es_tmax"] +
    eto["es_tmin"]
) / 2

In [6]:
eto["ea"] = (
    eto["es"] *
    eto["RH2M"] / 100
)

eto["vpd"] = eto["es"] - eto["ea"]

In [7]:
# Slope of vapour pressure curve
eto["delta"] = (4098 * saturation_vapor_pressure(eto["T2M"]) / ((eto["T2M"] + 237.3) ** 2))

In [8]:
# Psychrometric constant
eto["gamma"] = 0.000665 * eto["PS"]

In [9]:
# Extraterrestrial radiation
GSC = 0.0820

J = eto["day_of_year"]

eto["inverse_relative_distance"] = (
    1 +
    0.033 *
    np.cos(2 * np.pi * J / 365)
)

eto["solar_declination"] = (
    0.409 *
    np.sin(
        (2 * np.pi * J / 365) - 1.39
    )
)

eto["sunset_hour_angle"] = np.arccos(
    -np.tan(eto["latitude_rad"]) *
    np.tan(eto["solar_declination"])
)

eto["Ra"] = (
    (24 * 60 / np.pi)
    * GSC
    * eto["inverse_relative_distance"]
    * (
        eto["sunset_hour_angle"]
        * np.sin(eto["latitude_rad"])
        * np.sin(eto["solar_declination"])
        +
        np.cos(eto["latitude_rad"])
        * np.cos(eto["solar_declination"])
        * np.sin(eto["sunset_hour_angle"])))

In [10]:
# Clear-sky and shortwave radiation
eto["Rs"] = eto["ALLSKY_SFC_SW_DWN"]

# FAO near-sea-level approximation
eto["Rso"] = 0.75 * eto["Ra"]

eto["rs_rso_raw"] = (
    eto["Rs"] / eto["Rso"]
)

eto["radiation_ratio_review_flag"] = (
    eto["rs_rso_raw"] > 1
)

eto["rs_rso"] = eto["rs_rso_raw"].clip(
    lower=0,
    upper=1
)

eto["Rns"] = 0.77 * eto["Rs"]

In [12]:
# Net longwave radiation
SIGMA = 4.903e-9

tmax_k = eto["T2M_MAX"] + 273.16
tmin_k = eto["T2M_MIN"] + 273.16

eto["Rnl"] = (
    SIGMA
    *
    (
        (tmax_k ** 4 + tmin_k ** 4) / 2
    )
    *
    (
        0.34 -
        0.14 * np.sqrt(eto["ea"])
    )
    *
    (
        1.35 * eto["rs_rso"] -
        0.35
    )
)

eto["Rn"] = eto["Rns"] - eto["Rnl"]

In [13]:
# Calculate FAO-56 ET₀
G = 0

eto["ET0_mm_day"] = (
    (
        0.408
        * eto["delta"]
        * (eto["Rn"] - G)
        +
        eto["gamma"]
        * (900 / (eto["T2M"] + 273))
        * eto["WS2M"]
        * eto["vpd"]
    )
    /
    (
        eto["delta"]
        +
        eto["gamma"]
        * (
            1 +
            0.34 * eto["WS2M"]
        )
    )
)

In [14]:
# Inspect ET₀
eto["ET0_mm_day"].describe()

,ET0_mm_day
count,136002.000000
mean,4.868434
std,1.987112
min,0.852041
25%,3.071795
50%,4.797314
75%,6.575069
max,11.627580


In [15]:
print(
    "Missing ET0:",
    eto["ET0_mm_day"].isna().sum()
)

print(
    "Negative ET0:",
    (eto["ET0_mm_day"] < 0).sum()
)

print(
    "Radiation ratio > 1:",
    eto["radiation_ratio_review_flag"].sum()
)

Missing ET0: 0
Negative ET0: 0
Radiation ratio > 1: 0


In [16]:
eto[
    [
        "point_id",
        "obs_date",
        "ET0_mm_day",
        "T2M",
        "RH2M",
        "WS2M",
        "Rn"
    ]
].sort_values(
    "ET0_mm_day",
    ascending=False
).head(20)

,point_id,obs_date,ET0_mm_day,T2M,RH2M,WS2M,Rn
56028,grid_0016,2023-06-02,11.627580,32.30,38.78,5.33,14.424706
52449,grid_0015,2023-06-02,11.627580,32.30,38.78,5.33,14.424706
48870,grid_0014,2023-06-02,11.627580,32.30,38.78,5.33,14.424706
23817,grid_0007,2023-06-02,11.627306,32.30,38.78,5.33,14.423572
27396,grid_0008,2023-06-02,11.627306,32.30,38.78,5.33,14.423572
30975,grid_0009,2023-06-02,11.627306,32.30,38.78,5.33,14.423572
9501,grid_0003,2023-06-02,11.627029,32.30,38.78,5.33,14.422424
43820,grid_0013,2019-05-23,11.030690,32.64,32.25,3.55,14.804546
40241,grid_0012,2019-05-23,11.030690,32.64,32.25,3.55,14.804546
33083,grid_0010,2019-05-23,11.030690,32.64,32.25,3.55,14.804546


In [17]:
# Save ET
eto_output_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/eto_daily.csv.gz"
)

eto.to_csv(
    eto_output_path,
    index=False,
    compression="gzip"
)

print("ET0 dataset saved successfully.")
print(eto_output_path)

ET0 dataset saved successfully.
/content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/eto_daily.csv.gz


In [18]:
import os

print(
    "ET0 file exists:",
    os.path.exists(eto_output_path)
)

print("ET0 rows:", len(eto))

ET0 file exists: True
ET0 rows: 136002


In [19]:
from google.colab import files

files.download(eto_output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>